### Label all courses with BERT LLM

1. Load all courses
2. Extract topics using PreprocessLM (qwen off-the-shelf model)
3. Label topics using ClassifyLM (trained BERT model with custom threshold)
4. Merge labels into one 'pie chart' 

In [1]:
from utils.llm_utils import submit_message_LLM, topic_prompt 
# set seed 
from transformers import set_seed 
set_seed(42) 
import numpy as np 
import torch 
np.random.seed(42) 
torch.manual_seed(42) 
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
# gets saved at naiss2024-22-903/.cache/huggingface/models
from transformers import AutoModelForCausalLM, AutoTokenizer
def load_PreprocessLM(): 
    # Load Qwen2-7B-Instruct model. 
    API_KEY="<YOUR-APKI-KEY-HERE>"
    model_path="Qwen/Qwen2.5-7B-Instruct"
    device = "cuda" # the device to load the model onto
    
    model2 = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer2 = AutoTokenizer.from_pretrained(model_path)
    return model2, tokenizer2 

# Load trained BERT model 
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import sys
def load_ClassifyLM(): 
    sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")
    # load model from a working checkpoint 
    model_name='BERTv1final/checkpoint-430' #'distilbert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 
    model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                            problem_type="multi_label_classification",
                                                                            num_labels=9, dropout=0.2)
    model.eval()
    return model, tokenizer 

# misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))

/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/RL/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
#1. Load all courses from courses. 
import pandas as pd 
df = pd.read_excel('data/courses.ods', engine='odf')
#print(df)
print(df.keys())
uni = df.get(df.keys()[0])
print(uni)
course_names = df.get(df.keys()[3])
print(course_names)
course_descriptions = df.get(df.keys()[-1])

Index(['University ', 'Type ', 'Credits ', 'Course name ', 'Course code  ',
       'Learning objectives ', 'Course Description '],
      dtype='object')
0      LU 
1      LU 
2      LU 
3      LU 
4     KTH 
5     KTH 
6     KTH 
7     KTH 
8     KTH 
9     KTH 
10    KTH 
11    KTH 
12    KTH 
13    KTH 
14    KTH 
15    KTH 
16    KTH 
17    KTH 
18    KTH 
19    KTH 
20    KTH 
21    KTH 
22    KTH 
23    KTH 
24    NTU 
25    NTU 
26    NTU 
27    NTU 
28    NTU 
29    NTU 
30    NTU 
31    NTU 
32    NTU 
33    NTU 
34    NTU 
35    NTU 
36    NTU 
37    CMU 
38    CMU 
39    CMU 
40    CMU 
41    CMU 
42    CMU 
43    CMU 
44    CMU 
45    CMU 
46    CMU 
47    CMU 
48    CMU 
Name: University , dtype: object
0                                Advanced Web Security 
1                           Secure Systems Engineering 
2                                         Cryptography 
3                                         Web Security 
4     Theory and Methodology of Science (Natural an

In [4]:
university_dfs = dict(tuple(df.groupby(df.keys()[0])))
print(list(university_dfs.keys()))
# Take all KTH entries only 
#course_names = df.get(df.keys()[3])
course_names = university_dfs['KTH\xa0'].get(df.keys()[3])
course_names = course_names.reset_index()
course_names = course_names.get(df.keys()[3])
#course_descriptions = df.get(df.keys()[-1])
course_descriptions = university_dfs['KTH\xa0'].get(df.keys()[-1])
course_descriptions = course_descriptions.reset_index()
course_descriptions = course_descriptions.get(df.keys()[-1])
print(course_names)
print(course_descriptions[14])

['CMU\xa0', 'KTH\xa0', 'LU\xa0', 'NTU\xa0']
0     Theory and Methodology of Science (Natural and...
1     Theory of Science and Scientific methods in Cy...
2        The Cybersecurity Engineer's Role in Society  
3                              Cybersecurity Overview  
4          Cybersecurity in a Socio-Technical Context  
5                                Applied Cryptography  
6                                     Ethical Hacking  
7                          Foundations of Cryptography 
8                      Privacy Enhancing Technologies  
9                   Project course in System Security  
10                            Language-Based Security  
11    Cyber-Physical Security in Time-Critical Syste...
12                         Networked Systems Security  
13                Advanced Networked Systems Security  
14                Building Networked Systems Security  
15            Digital forensics and incident response  
16    Security Analysis of Large-Scale Computer Syst...
17  

In [12]:
# case example 
model, tokenizer = load_ClassifyLM()
strings = ['building networked systems security', 'handling contemporary security problems for networked systems', 
          'problem-based learning','teamwork in cybersecurity','investigating requirements for networked systems security',
          'designing specifications for cybersecurity', 'preparing solutions with professional tools',
           'critically assessing the efficiency of alternative solutions']
tokenized_course = {}
for i in range(len(strings)): 
        #print(course[str(i)])
        tokenized_course[str(i)] = tokenizer(strings[i], return_tensors='pt') #, padding='max_length', max_length=64)
        with torch.no_grad(): 
            predictions= model(**tokenized_course[str(i)])
            prediction = sigmoid(predictions.logits).detach().cpu().numpy()
            predictions = (prediction > 0.5).astype(int).reshape(-1)
            
            if sum(predictions) == 0: 
                print('empty')
                out = np.argmax(prediction)
                prediction_zeros= np.zeros(predictions.shape)
                prediction_zeros[out] = 1 
                predictions = prediction_zeros.astype(int)

            #llm_labels.append(predictions)
            print([i for i,x in enumerate(predictions) if x >0])

empty
[7]
empty
[4]
[0]
[7]
empty
[5]
[7]
[7]
[7]


In [15]:
strings = ["Ethical Hacking",
    "Network and vulnerability scanning",
    "Exploit development platforms",
    "Command and control",
    "Password cracking",
    "Independent attack project",
    "Virtual environment setup",
    "Language-Based Security",
    "Fundamental principles, models and concepts for computer security",
    "Software security by information flow control",
    "Web application and database security",
    "Security for mobile applications",
    "Hot topics in computer security",
    "State-of-the-art in programming language for security",
    "Building Networked Systems Security",
    "Handling contemporary security problems for networked systems",
    "Problem-based learning",
    "Teamwork in cybersecurity",
    "Investigating requirements for networked systems security",
    "Designing specifications for cybersecurity",
    "Preparing solutions with professional tools",
    "Critically assessing the efficiency of alternative solutions",
    "Network Security",
    "Introduction to Networking",
    "Security in Networks",
    "Network Programming",
    "Packet Trace Analysis with Wireshark",
    "TLS and Security in Networked Systems",
    "Software Security",
    "Challenges in software security",
    "Principles of secure software development",
    "Mechanisms for secure coding",
    "Tools for software security",
    "Causes of vulnerabilities",
    "Strategies to avoid vulnerabilities",
    "Defenses against attacks",
    "Secure programming practices",
    "Language-specific secure coding",
    "System-level defenses",
    "Architectural approaches for security",
    "Run-time enforcement techniques",
    "Software analysis tools",
    "Vulnerability detection methods",
    "Application of security tools in scenarios",
    "Privacy Preserving Technologies & Security in AI",
    "Fully Homomorphic Encryption",
    "Secure Multi-computation",
    "Searchable Encryption",
    "Data Sharing in Machine Learning",
    "Advanced Real-World Networks",
    "4G and 5G network infrastructures",
    "IPv6",
    "SDN and VFN",
    "Data centers",
    "Mesh networks",
    "Embedded networks",
    "Security in Networked Systems",
    "Network and transport-layer attacks and defenses",
    "Network intrusion detection",
    "Denial of service (DoS) and distributed denial-of-service (DDoS) detection and reaction",
    "Worm and virus propagation",
    "Tracing the source of attacks",
    "Traffic analysis",
    "Techniques for hiding the source or destination of network traffic",
    "Secure routing protocols",
    "Content poisoning attacks",
    "Advanced techniques for reacting to network attacks",
    "Information Security Policy and Management",
    "Security marketplace overview",
    "Decision making with multiple parties involved",
    "Role of policy in information security",
    "Intra-organization policies",
    "Market and competition impact on security provision",
    "Causes of market failure (externalities)",
    "Policy tools to mitigate market failure",
    "Key laws and regulations on product liability and security standards",
    "Overview of security industry trends, technologies, vendor and user strategies",
    "Managerial and policy issues in information security provision",
    "When and how policy intervention is needed"
]
for i in range(len(strings)): 
        print(strings[i])
        tokenized_course[str(i)] = tokenizer(strings[i], return_tensors='pt') #, padding='max_length', max_length=64)
        with torch.no_grad(): 
            predictions= model(**tokenized_course[str(i)])
            prediction = sigmoid(predictions.logits).detach().cpu().numpy()
            predictions = (prediction > 0.5).astype(int).reshape(-1)
            
            if sum(predictions) == 0: 
               # print('empty')
                out = np.argmax(prediction)
                prediction_zeros= np.zeros(predictions.shape)
                prediction_zeros[out] = 1 
                predictions = prediction_zeros.astype(int)

            #llm_labels.append(predictions)
            print([i for i,x in enumerate(predictions) if x >0])

Ethical Hacking
[8]
Network and vulnerability scanning
[4]
Exploit development platforms
[2]
Command and control
[0]
Password cracking
[6]
Independent attack project
[5]
Virtual environment setup
[4]
Language-Based Security
[5]
Fundamental principles, models and concepts for computer security
[5]
Software security by information flow control
[2]
Web application and database security
[7]
Security for mobile applications
[7]
Hot topics in computer security
[7]
State-of-the-art in programming language for security
[7]
Building Networked Systems Security
[7]
Handling contemporary security problems for networked systems
[4]
Problem-based learning
[0]
Teamwork in cybersecurity
[7]
Investigating requirements for networked systems security
[5]
Designing specifications for cybersecurity
[7]
Preparing solutions with professional tools
[7]
Critically assessing the efficiency of alternative solutions
[7]
Network Security
[7]
Introduction to Networking
[4]
Security in Networks
[7]
Network Programmi

In [13]:
# case example 
strings = ['ethical hacking', 'network and vulnerability scanning', 'exploit development platforms', 'command and control', 'password cracking', 'independent attack project', 'virtual environment setup']

for i in range(len(strings)): 
        #print(course[str(i)])
        tokenized_course[str(i)] = tokenizer(strings[i], return_tensors='pt') #, padding='max_length', max_length=64)
        with torch.no_grad(): 
            predictions= model(**tokenized_course[str(i)])
            prediction = sigmoid(predictions.logits).detach().cpu().numpy()
            predictions = (prediction > 0.5).astype(int).reshape(-1)
            
            if sum(predictions) == 0: 
                print('empty')
                out = np.argmax(prediction)
                prediction_zeros= np.zeros(predictions.shape)
                prediction_zeros[out] = 1 
                predictions = prediction_zeros.astype(int)

            #llm_labels.append(predictions)
            print([i for i,x in enumerate(predictions) if x >0])

[8]
[4]
[2]
[0]
[6]
empty
[5]
empty
[4]


In [5]:
#2. Extract topics using PreprocessLM (qwen off-the-shelf model)
import re
from datasets import Dataset 
from utils.load_data import clean_text
import torch
import numpy as np

# load PreprocessLM model 
model2, tokenizer2 = load_PreprocessLM()
model, tokenizer = load_ClassifyLM()
# loop over all courses: 

device = 'cuda:0'
KA_labels = [] 
for idx in range(len(course_descriptions)): 
    print(idx)
    description = course_descriptions[idx]                             # read course descriptions
    topic = course_names[idx]                                          # read course topics 
    print('name: ',topic)
    #print('description: ', description)
    #lo = LO[idx]                                                       # read learning objectives 
    message = topic_prompt(topic, description)                         # create prompt based on topic + description
    response = submit_message_LLM(model2, message, tokenizer2, device)  # put the formatted prompt through the LLM 

    #print('response: ', response)
    subtopic = response.split('- ')[1:] # isolates subtopics 
    subtopics = {}
    for i, sub in enumerate(subtopic): 
        # if key does not exist, create it, otherwise append? 
        #print(clean_text(sub))
        print(sub)
        subtopics[str(i)] = [clean_text(sub)]

    # make them into the right format (a dataset?) 
    course = Dataset.from_dict(subtopics)
    llm_labels = [] 
    tokenized_course = {}
    for i in range(len(course.features)): 
        #print(course[str(i)])
        tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
        with torch.no_grad(): 
            predictions= model(**tokenized_course[str(i)])
            prediction = sigmoid(predictions.logits).detach().cpu().numpy()
            predictions = (prediction > 0.5).astype(int).reshape(-1)
            
            if sum(predictions) == 0: 
                print('empty')
                out = np.argmax(prediction)
                prediction_zeros= np.zeros(predictions.shape)
                prediction_zeros[out] = 1 
                predictions = prediction_zeros.astype(int)

            llm_labels.append(predictions)
            print([i for i,x in enumerate(predictions) if x >0])
    # combine all labels of the course. 
    llm_labels2 = np.array(llm_labels)
    
    final_output = np.sum(llm_labels2,axis=0) / (np.sum(llm_labels2)) #/ len(course.features) # divide by the total number of 'items' 

    # now compare this to feeding just the course name 
    with torch.no_grad(): 
        predictions = model(**tokenizer(clean_text(topic), return_tensors='pt'))
        predictionn = sigmoid(predictions.logits.detach().cpu().numpy())
        predictions = (prediction > 0.5).astype(int).reshape(-1)
        if sum(predictions) == 0: 
                print('empty')
                out = np.argmax(prediction)
                prediction_zeros= np.zeros(predictions.shape)
                prediction_zeros[out] = 1 
                predictions = prediction_zeros.astype(int)
        #print('course name: ',predictions)
        print([i for i,x in enumerate(predictions) if x >0])
        llm_labels.append(predictions)
    print(predictions)
    llm_labels = np.array(llm_labels)
    final_output2 = np.sum(llm_labels,axis=0) / (np.sum(llm_labels)) #/ len(course.features) # divide by the total number of 'items' 
    print('LLM Labels: ',final_output2)
    KA_labels.append(final_output2) 
    print([KAs[str(j.item())] for j in np.where(final_output2>0)[0]])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

0
name:  Theory and Methodology of Science (Natural and Technological Science)  


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Scientific knowledge

Definitions

Hypothesis testing

Observations and measurements

Experiments

Models

Statistical reasoning

Causes and explanations

Engineering design

Qualitative methods

Research ethics

Risk and risk assessment
[0]
[5]
[2, 3]
[0]
[0]
[5]
[7, 8]
[5]
[2, 3]
[5]
[8]
[7]
[7]
[0 0 0 0 0 0 0 1 0]
LLM Labels:  [0.1875 0.     0.125  0.125  0.     0.25   0.     0.1875 0.125 ]
['miscellaneous', 'software security', 'component security', 'system security', 'organizational security', 'societal security']
1
name:  Theory of Science and Scientific methods in Cybersecurity 
Scientific methodology in cybersecurity

Gender equality, diversity, and equal conditions in cybersecurity

Sustainability in cybersecurity

Ethical dilemmas in cybersecurity

Application of scientific methods in cybersecurity analysis
[7, 8]
[6, 8]
[7, 8]
[8]
[7]
[7]
[0 0 0 0 0 0 0 1 0]
LLM Labels:  [0.         0.         0.         0.         0.         0.
 0.11111111 0.44444444 0.44444444]
['human sec

In [23]:
#CMU / overall 
# combine into two pie charts, one for mandatory and one for elective courses. 
KA = np.array([KA_labels for i, KA_labels in enumerate(KA_labels) if i in [0, 5, 7, 9,10,11]]) #CMU sample MSIT-IS curriculum
credits = np.array(university_dfs['CMU\xa0'].get(df.keys()[2]).values).reshape(-1, 1)
credits = [K for i, K in enumerate(credits) if i in [0, 5, 7, 9,10,11]]
combined = np.sum(credits * KA, axis=0)
for i in [0, 5, 7, 9,10, 11]: 
    print(course_names[i])
    print(course_descriptions[i])
    print(KA_labels[i])
    #print(credits[i])
print(credits)
print('overall')
print(100*combined / np.sum(combined))

Fundamentals of Telecommunications and Computer Networks 
14-740 is a graduate-level, first-course in computer and telecommunication networks. There is no pre-requisite of an undergraduate equivalent, but basic computer, programming and probability theory background is required. The primary objective of this course is for you to learn the fundamental principles underlying computer and telecommunication networks. Using a top-down approach, we will cover topics in the application, transport, network and link layers of the protocol stack. We will also go over advanced topics, including network management, traffic engineering, and router internals. Besides learning about the nuts and bolts, you will gain an understanding as well in engineering tradeoffs made and design principles used in computer and telecommunication networks. Another objective is for you to apply some of this knowledge in the context of systems projects. We will follow an aggressive pace in this course. 
[0.25       0.  

In [11]:
#NTU
KA_labels = np.array(KA_labels)

types = university_dfs['NTU\xa0'].get(df.keys()[1])
types = types.reset_index(drop=True) 
print(types)
# Get indices for 'Mandatory'
# Strip whitespace before comparison
mandatory_indices = types[types.str.strip() == 'Mandatory' ].index.tolist()
conditionally_elective_indices = types[types.str.strip() == 'elective'].index.tolist() #NTU
print("Mandatory indices:", mandatory_indices)
print("Conditionally Elective indices:", conditionally_elective_indices)

combined = np.sum(KA_labels[mandatory_indices],axis=0)
print('mandatory')
print(100*combined / np.sum(combined))
combined = np.sum(KA_labels[conditionally_elective_indices],axis=0)
print('mandatory')
print(100*combined / np.sum(combined))

0     Mandatory 
1     Mandatory 
2     Mandatory 
3     Mandatory 
4      elective 
5      elective 
6      elective 
7      elective 
8      elective 
9      elective 
10     elective 
11     elective 
12     elective 
Name: Type , dtype: object
Mandatory indices: [0, 1, 2, 3]
Conditionally Elective indices: [4, 5, 6, 7, 8, 9, 10, 11, 12]
mandatory
[ 3.84615385 27.82828283  9.75135975  0.          5.          8.8966589
 14.2968143  26.53457653  3.84615385]
mandatory
[ 5.00593542 14.72222222 12.46111913  5.33810325 16.30320297 14.18690877
  4.20940171 24.67481592  3.0982906 ]


In [6]:
#KTH
#KA = np.array([K for i, K in enumerate(KA_labels) if i in [0, 5, 7, 8,11]]) #CMU sample MSIT-IS curriculum
KA_labels = np.array(KA_labels)

types = university_dfs['KTH\xa0'].get(df.keys()[1])
types = types.reset_index(drop=True) 
print(types)
# Get indices for 'Mandatory'
# Strip whitespace before comparison
mandatory_indices = types[types.str.strip() == 'Mandatory' ].index.tolist()
conditionally_elective_indices = types[types.str.strip() == 'Conditionally Elective'].index.tolist() #KTH
#conditionally_elective_indices = types[types.str.strip() == 'elective'].index.tolist() #NTU
print("Mandatory indices:", mandatory_indices)
print("Conditionally Elective indices:", conditionally_elective_indices)

combined = np.sum(KA_labels[mandatory_indices],axis=0)
print('mandatory before taking credits into account')
print(100*combined / np.sum(combined))
combined = np.sum(KA_labels[conditionally_elective_indices],axis=0)
print('elective')
print(100*combined / np.sum(combined))

credits = university_dfs['KTH\xa0'].get(df.keys()[2]).values
print(credits)
print(credits[mandatory_indices])
combined = np.sum(credits[mandatory_indices].reshape(-1, 1) * KA_labels[mandatory_indices], axis=0)

for i in mandatory_indices: 
    print('name: ',course_names[i])
    print('description: ',course_descriptions[i])
    print('KA: ',KA_labels[i])
#print('mandatory')
#print(credits)
#print(mandatory_indices)
print(100*combined / np.sum(combined))

0                  Mandatory 
1                  Mandatory 
2                  Mandatory 
3                  Mandatory 
4                  Mandatory 
5                  Mandatory 
6                  Mandatory 
7     Conditionally Elective 
8     Conditionally Elective 
9     Conditionally Elective 
10    Conditionally Elective 
11    Conditionally Elective 
12    Conditionally Elective 
13    Conditionally Elective 
14    Conditionally Elective 
15    Conditionally Elective 
16    Conditionally Elective 
17    Conditionally Elective 
18    Conditionally Elective 
19    Conditionally Elective 
Name: Type , dtype: object
Mandatory indices: [0, 1, 2, 3, 4, 5, 6]
Conditionally Elective indices: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
mandatory before taking credits into account
[ 6.83441558 15.00721501  4.38311688  4.38311688  3.8961039  13.13131313
  7.25108225 24.27308802 20.84054834]
elective
[ 6.7950576  20.38562404 12.76031215  2.98327759 12.81642512 11.53548867
  5.66544567

In [7]:
for i in conditionally_elective_indices: 
    print('name: ',course_names[i])
    print(i)
    #print('description: ',course_descriptions[i])
    print('KA: ',100*KA_labels[i])

name:  Foundations of Cryptography 
7
KA:  [  0. 100.   0.   0.   0.   0.   0.   0.   0.]
name:  Privacy Enhancing Technologies  
8
KA:  [ 0.         21.42857143  0.          0.          0.          0.
 21.42857143 28.57142857 28.57142857]
name:  Project course in System Security  
9
KA:  [ 8.69565217  0.         30.43478261 26.08695652  8.69565217  8.69565217
  0.         17.39130435  0.        ]
name:  Language-Based Security  
10
KA:  [ 0.          0.         44.44444444  0.          0.          0.
 22.22222222 33.33333333  0.        ]
name:  Cyber-Physical Security in Time-Critical Systems  
11
KA:  [ 4. 36.  4.  4. 12. 16.  0. 20.  4.]
name:  Networked Systems Security  
12
KA:  [12.5  0.   0.   0.  50.  37.5  0.   0.   0. ]
name:  Advanced Networked Systems Security  
13
KA:  [10.  0. 20.  0. 40. 20.  0. 10.  0.]
name:  Building Networked Systems Security  
14
KA:  [ 0.   0.   0.   0.  25.  12.5  0.  62.5  0. ]
name:  Digital forensics and incident response  
15
KA:  [ 0.        

In [11]:
# KTH selected program (for figure in case example) 
selected = [8, 12, 13, 14]
credits = university_dfs['KTH\xa0'].get(df.keys()[2]).values
types = university_dfs['KTH\xa0'].get(df.keys()[1])
types = types.reset_index(drop=True) 
# Get indices for 'Mandatory'
# Strip whitespace before comparison
mandatory_indices = types[types.str.strip() == 'Mandatory' ].index.tolist()
total_program = mandatory_indices + selected 
combined = np.sum(credits[total_program].reshape(-1, 1) * KA_labels[total_program], axis=0)

print('total program: ',total_program)
#print(credits)
print('total number of credits: ',sum(credits[total_program]))
#print(mandatory_indices)
print(100*combined / np.sum(combined))

total program:  [0, 1, 2, 3, 4, 5, 6, 8, 12, 13, 16]
total number of credits:  69.5
[ 9.1794746  12.76952464  4.92969261  2.77141923 13.85437105 15.68472495
  5.31185441 22.72798644 12.77095207]


In [10]:
# KTH selected program (for figure in case example) 
selected = [8, 12, 13, 14]
credits = university_dfs['KTH\xa0'].get(df.keys()[2]).values
types = university_dfs['KTH\xa0'].get(df.keys()[1])
types = types.reset_index(drop=True) 
# Get indices for 'Mandatory'
# Strip whitespace before comparison
mandatory_indices = types[types.str.strip() == 'Mandatory' ].index.tolist()
total_program = mandatory_indices + selected 
combined = np.sum(credits[total_program].reshape(-1, 1) * KA_labels[total_program], axis=0)

print('total program: ',total_program)
#print(credits)
print('total number of credits: ',sum(credits[total_program]))
#print(mandatory_indices)
print(100*combined / np.sum(combined))

total program:  [0, 1, 2, 3, 4, 5, 6, 8, 12, 13, 14]
total number of credits:  69.5
[ 6.78139307 12.76952464  4.92969261  2.77141923 15.35317201 14.63556428
  5.31185441 24.67642769 12.77095207]


In [9]:
# KTH selected program (for figure in case example) 
selected = [11, 12, 13, 14]
credits = university_dfs['KTH\xa0'].get(df.keys()[2]).values
types = university_dfs['KTH\xa0'].get(df.keys()[1])
types = types.reset_index(drop=True) 
# Get indices for 'Mandatory'
# Strip whitespace before comparison
mandatory_indices = types[types.str.strip() == 'Mandatory' ].index.tolist()
total_program = mandatory_indices + selected 
combined = np.sum(credits[total_program].reshape(-1, 1) * KA_labels[total_program], axis=0)

print('total program: ',total_program)
#print(credits)
print('total number of credits: ',sum(credits[total_program]))
#print(mandatory_indices)
print(100*combined / np.sum(combined))

total program:  [0, 1, 2, 3, 4, 5, 6, 11, 12, 13, 14]
total number of credits:  69.5
[ 7.21304774 14.34198096  5.36134729  3.2030739  16.64813604 16.36218298
  2.99941865 23.75145338 10.11935906]


In [162]:
# KTH other run 
# mandatory
[ 7.65966628 18.39918169  4.87629459  4.87629459  5.17836594 12.46004347
  5.27745813 24.22724076 17.04545455]
# elective 
[ 6.7950576  20.38562404 12.76031215  2.98327759 12.81642512 11.53548867
  5.66544567 23.01441312  4.04395604]


[10.50057537 21.97161488  5.22151899  1.42405063  5.52359033 10.63674722
  3.2853855  21.43747603 19.99904104]

0                  Mandatory 
1                  Mandatory 
2                  Mandatory 
3                  Mandatory 
4                  Mandatory 
5                  Mandatory 
6                  Mandatory 
7     Conditionally Elective 
8     Conditionally Elective 
9     Conditionally Elective 
10    Conditionally Elective 
11    Conditionally Elective 
12    Conditionally Elective 
13    Conditionally Elective 
14    Conditionally Elective 
15    Conditionally Elective 
16    Conditionally Elective 
17    Conditionally Elective 
18    Conditionally Elective 
19    Conditionally Elective 
Name: Type , dtype: object
Mandatory indices: [0, 1, 2, 3, 4, 5, 6]
Conditionally Elective indices: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
mandatory before taking credits into account
[10.10822511 17.22943723  4.64285714  1.78571429  4.15584416 10.90909091
  5.34632035 22.67316017 23.14935065]
elective
[ 7.35887301 20.98595849 10.86178314  3.91034087 15.90588156  7.30725796
  3.57142857

In [99]:
credits = university_dfs['KTH\xa0'].get(df.keys()[2]).values
print(credits)
print(credits[mandatory_indices])
combined = np.sum(credits[mandatory_indices].reshape(-1, 1) * KA_labels[mandatory_indices], axis=0)

for i in mandatory_indices: 
    print('name: ',course_names[i])
    print('description: ',course_descriptions[i])
    print('KA: ',KA_labels[i])
#print('mandatory')
#print(credits)
#print(mandatory_indices)
print(100*combined / np.sum(combined))

[4.5 3 2 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5 7.5
 7.5]
[4.5 3 2 7.5 7.5 7.5 7.5]
name:  Theory and Methodology of Science (Natural and Technological Science)  
description:  The following is an incomplete list of topics covered in the course. Scientific knowledge Definitions Hypothesis testing Observations and measurements Experiments Models Statistical reasoning Causes and explanations Engineering design Qualitative methods Research ethics Risk and risk assessment  
KA:  [0.25  0.    0.125 0.125 0.    0.25  0.    0.125 0.125]
name:  Theory of Science and Scientific methods in Cybersecurity 
description:  The course highlights, how different parts of the scientific methodology are relevant for cybersecurity in different situations. The focus is to analyse how scientific methods influence our knowledge of issues in cybersecurity, including their relation to different aspects of the subjects gender equality, diversity and equal conditions sustainability, ethical d

### Valtteri's and Sara's KDs vs course topics 
We use Cohen's kappa for now, because they're just two annotators. 

In [1]:
import pandas as pd 
df = pd.read_excel('data/Topics_Annotation.xlsx')
print(df)

                                                Topic Annotator 1  \
0                                     Ethical Hacking         1,6   
1                  Network and vulnerability scanning           4   
2                       Exploit development platforms         2,7   
3                                 Command and control           0   
4                                   Password cracking           1   
..                                                ...         ...   
74            Policy tools to mitigate market failure         0,5   
75  Key laws and regulations on product liability ...           5   
76  Overview of security industry trends, technolo...           0   
77  Managerial and policy issues in information se...         5,7   
78         When and how policy intervention is needed         NAN   

   Annotator 2  Annotator 3               Sara  Valtteri         Paul  \
0           6,8        2,5,6  0,1,2,3,4,5,6,7,8    2,7,8  3,2,4,5,6,0   
1       3,4,5,7          

In [125]:
# also run the topics through LLM2. 
# Load trained DistilBERT model 
from sklearn.metrics import multilabel_confusion_matrix
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import sys
import torch
import numpy as np
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")

# load model from a working checkpoint 
model_name='BERTv1final/checkpoint-430' #'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 
# misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inplace=False)


In [126]:
# convert df to Dataset 
from datasets import Dataset
df = pd.read_excel('data/Topics_Annotation.xlsx')
print(df)
SV_data = df.drop([df.columns[1],df.columns[2],df.columns[3]], axis=1).astype(str)
print(SV_data)
sv_topics = SV_data.drop([SV_data.columns[1],SV_data.columns[2]], axis=1).astype(str)
print(df.keys())
course_descriptions = df.get(df.keys()[0])
print(course_descriptions)
print(course_descriptions[0])
sv_dataset = Dataset.from_pandas(sv_topics)

                                                Topic Annotator 1  \
0                                     Ethical Hacking         1,6   
1                  Network and vulnerability scanning           4   
2                       Exploit development platforms         2,7   
3                                 Command and control           0   
4                                   Password cracking           1   
..                                                ...         ...   
74            Policy tools to mitigate market failure         0,5   
75  Key laws and regulations on product liability ...           5   
76  Overview of security industry trends, technolo...           0   
77  Managerial and policy issues in information se...         5,7   
78         When and how policy intervention is needed         NAN   

   Annotator 2  Annotator 3               Sara  Valtteri         Paul  \
0           6,8        2,5,6  0,1,2,3,4,5,6,7,8    2,7,8  3,2,4,5,6,0   
1       3,4,5,7          

In [131]:
import re
from datasets import Dataset 
from utils.load_data import clean_text
device = 'cuda:0'
KA_labels = [] 

# make them into the right format (a dataset?) 
th = 0.5 
llm_labels = [] 
llm_labelsN = [] 
tokenized_course = {}
for i in range(len(sv_dataset)): 
    print(course_descriptions[i])
    tokenized_course[str(i)] = tokenizer(course_descriptions[i], return_tensors='pt') #, padding='max_length', max_length=64)
    with torch.no_grad(): 
        predictions= model(**tokenized_course[str(i)])
        prediction = sigmoid(predictions.logits).detach().cpu().numpy()
        predictions = (prediction > th).astype(int).reshape(-1)
        llm_labels.append(predictions)
        print(predictions)
        out = [idx for idx,x in enumerate(predictions) if x >0]
        if sum(predictions) == 0: 
            print('empty')
            out = [np.argmax(prediction)]
            print(prediction)
            #print(out)
        llm_labelsN.append(out)
        print(out)
# combine all labels of the course. 
llm_labels2 = np.array(llm_labels)
#print(llm_labels2)
final_output = np.sum(llm_labels2,axis=0) / (np.sum(llm_labels2)) #/ len(course.features) # divide by the total number of 'items' 

print('LLM Labels: ',final_output)
KA_labels.append(final_output) 
#print([KAs[str(j.item())] for j in np.where(final_output2>0)[0]])

Ethical Hacking
[0 0 0 0 0 0 0 0 1]
[8]
Network and vulnerability scanning
[0 0 0 0 1 0 0 0 0]
[4]
Exploit development platforms
[0 0 1 0 0 0 0 0 0]
[2]
Command and control
[1 0 0 0 0 0 0 0 0]
[0]
Password cracking
[0 0 0 0 0 0 1 0 0]
[6]
Independent attack project
[0 0 0 0 0 0 0 0 0]
empty
[[0.00329785 0.242468   0.0908689  0.0636442  0.10814124 0.4024659
  0.01209703 0.00703111 0.00512737]]
[5]
Virtual environment setup
[0 0 0 0 0 0 0 0 0]
empty
[[0.00302501 0.02220791 0.27622366 0.1882631  0.40680858 0.17617433
  0.00878089 0.01465225 0.00614454]]
[4]
Language-Based Security
[0 0 0 0 0 0 0 0 0]
empty
[[0.00860591 0.12935835 0.12757318 0.03051371 0.00342743 0.4574951
  0.0584657  0.0817697  0.00816038]]
[5]
Fundamental principles, models and concepts for computer security
[0 0 0 0 0 0 0 0 0]
empty
[[0.0315569  0.01426353 0.03940666 0.01853545 0.02978416 0.34658897
  0.00446201 0.06257322 0.00185883]]
[5]
Software security by information flow control
[0 0 1 0 0 0 0 0 0]
[2]
Web applic

In [143]:
print(llm_labelsN)
row_data = [','.join(map(str, sublist)) for sublist in llm_labelsN]
print(row_data)
row_data2 = []
for x in row_data: 
    if len(x) == 0: 
        row_data2.append('NAN')
    else: 
        row_data2.append(x)
#row_data2 = ['NAN' for x in row_data if len(x)==0 else x]
print(row_data2)
df['LLM'] = row_data2
print(df)
print(df['LLM'][:])

[[8], [4], [2], [0], [6], [5], [4], [5], [5], [2], [7], [7], [7], [7], [7], [4], [0], [7], [5], [7], [7], [7], [7], [4], [7], [7], [4], [4], [2], [2], [2], [2], [2], [7], [7], [5], [2], [2], [5], [2], [7], [2], [2, 3], [7], [1, 6, 7, 8], [1], [1], [1], [4], [4], [4], [4], [4], [1], [4], [4], [7], [4], [4], [4], [1], [1], [7], [4], [4], [1], [4], [7], [7], [7, 8], [7, 8], [7], [7], [7], [7, 8], [7], [7], [7], [7, 8]]
['8', '4', '2', '0', '6', '5', '4', '5', '5', '2', '7', '7', '7', '7', '7', '4', '0', '7', '5', '7', '7', '7', '7', '4', '7', '7', '4', '4', '2', '2', '2', '2', '2', '7', '7', '5', '2', '2', '5', '2', '7', '2', '2,3', '7', '1,6,7,8', '1', '1', '1', '4', '4', '4', '4', '4', '1', '4', '4', '7', '4', '4', '4', '1', '1', '7', '4', '4', '1', '4', '7', '7', '7,8', '7,8', '7', '7', '7', '7,8', '7', '7', '7', '7,8']
['8', '4', '2', '0', '6', '5', '4', '5', '5', '2', '7', '7', '7', '7', '7', '4', '0', '7', '5', '7', '7', '7', '7', '4', '7', '7', '4', '4', '2', '2', '2', '2', '2', '7

In [144]:
# Merge annotator 1, 2, and 3. 
import re 
from collections import Counter
def return_most_frequent(labels_combined): 
    numbers = re.findall(r'\b\d+(?:\.\d+)?\b', labels_combined) 
    counter = Counter(numbers)
    max_frequency = max(counter.values())
    most_frequent = [num for num, count in counter.items() if count == max_frequency]
    # format the output as a string 
    if len(most_frequent) == 1: 
        most_frequent = most_frequent[0]
    else: 
        most_frequent = ",".join(most_frequent)
    return most_frequent

# merge annotator 1, 2, and 3 using the label that occurs most often 
merged = [] 
ann1_data = df['Annotator 1']
ann2_data = df['Annotator 2 ']
ann3_data = df['Annotator 3 ']
for i,ele in enumerate(ann1_data): 
    # leave out any NAN 
    if ann1_data[i] == 'NAN': 
        ann1 = ''
    else: 
        ann1 = str(ann1_data[i])
    if ann2_data[i] == 'NAN': 
        ann2 = ''
    else: 
        ann2 = str(ann2_data[i])
    if ann3_data[i] =='NAN': 
        ann3 = ''
    else: 
        ann3 = str(ann3_data[i])
    # combine labels 
    labels_combined = ann1+',' + ann2 +','+ ann3 
    # choose the maximum only, if multiple have the same number of occurrences, choose both 
    most_frequent = return_most_frequent(labels_combined)
    #print(most_frequent)
    merged.append(most_frequent)

df['merged'] = merged
#print(df['merged'])

In [145]:
# compute agreement like Valtteri and such have done 
import re
from itertools import combinations

def extract_numbers(text):
    """Extract all numbers from a string as a set"""
    return set(re.findall(r'\b\d+(?:\.\d+)?\b', text))

def count_positional_overlaps(lists_of_strings, list_names):
    """
    Count how many positions have overlapping numbers between pairs of lists
    
    Args:
        lists_of_strings: List of 4 lists containing strings in order: 
                         llm_data, S_data, V_data, swits_data
    """
    #if len(lists_of_strings) != 4:
     #   raise ValueError("Expected exactly 4 lists of strings")
    
    # Verify all lists have the same length
    list_lengths = [len(lst) for lst in lists_of_strings]
    if len(set(list_lengths)) > 1:
        raise ValueError("All lists must have the same length")
    
    n_samples = len(lists_of_strings[0])
    
    # Precompute number sets for all positions
    number_sets = []
    for lst in lists_of_strings:
        list_sets = [extract_numbers(s) for s in lst]
        number_sets.append(list_sets)
    
    results = {}
    
    # Check all pair-wise combinations of lists
    for i, j in combinations(range(len(list_names)), 2):
        count = 0
        for pos in range(n_samples):
            set_i = number_sets[i][pos]
            set_j = number_sets[j][pos]
            
            # Check if both strings have numbers and there's overlap
            if set_i and set_j and set_i.intersection(set_j):
                count += 1
        
        pair_name = f"{list_names[i]}_vs_{list_names[j]}"
        results[pair_name] = count
    
    return results

# Define list names in the correct order
list_names = ["llm_data", "S_data", "V_data", "P_data","swits_data"]
llm_data = [str(x) for x in df['LLM'].tolist()]
S_data = [str(x) for x in df['Sara '].tolist()]
V_data = [str(x) for x in df['Valtteri'].tolist()]
P_data = [str(x) for x in df['Paul'].tolist()]
swits_data = [str(x) for x in df['merged'].tolist()]
lists = [llm_data, S_data, V_data, P_data,swits_data]
result = count_positional_overlaps(lists, list_names)

print("Position-wise overlapping counts:")
print("=" * 50)
for pair, count in result.items():
    print(f"{pair}: {100*count/len(llm_data)} positions")

print("=" * 50)
print(f"Total samples analyzed: {len(llm_data)}")

Position-wise overlapping counts:
llm_data_vs_S_data: 68.35443037974683 positions
llm_data_vs_V_data: 44.30379746835443 positions
llm_data_vs_P_data: 70.88607594936708 positions
llm_data_vs_swits_data: 58.22784810126582 positions
S_data_vs_V_data: 70.88607594936708 positions
S_data_vs_P_data: 87.34177215189874 positions
S_data_vs_swits_data: 64.55696202531645 positions
V_data_vs_P_data: 74.68354430379746 positions
V_data_vs_swits_data: 59.49367088607595 positions
P_data_vs_swits_data: 78.48101265822785 positions
Total samples analyzed: 79


In [135]:
def count_positional_overlaps(lists_of_strings, list_names):
    """
    Count positional overlaps between pairs of lists with three categories:
    - Full overlap: All numbers match
    - Partial overlap: Some but not all numbers match
    - No overlap: No numbers match
    
    Args:
        lists_of_strings: List of lists containing strings
        list_names: Names of the lists for labeling
    """
    # Verify all lists have the same length
    list_lengths = [len(lst) for lst in lists_of_strings]
    if len(set(list_lengths)) > 1:
        raise ValueError("All lists must have the same length")
    
    n_samples = len(lists_of_strings[0])
    
    # Precompute number sets for all positions
    number_sets = []
    for lst in lists_of_strings:
        list_sets = [extract_numbers(s) for s in lst]
        number_sets.append(list_sets)
    
    results = {}
    
    # Check all pair-wise combinations of lists
    for i, j in combinations(range(len(list_names)), 2):
        full_overlap = 0
        partial_overlap = 0
        no_overlap = 0
        
        for pos in range(n_samples):
            set_i = number_sets[i][pos]
            set_j = number_sets[j][pos]
            
            # Cases where one or both sets are empty
            if not set_i or not set_j:
                if not set_i and not set_j:
                    # Both empty - consider as no overlap since no numbers to compare
                    no_overlap += 1
                else:
                    # One empty, one has numbers - no overlap
                    no_overlap += 1
                continue
            
            # Both sets have numbers
            intersection = set_i.intersection(set_j)
            
            if len(intersection) == 0:
                # No numbers in common
                no_overlap += 1
            elif set_i == set_j:
                # All numbers match exactly
                full_overlap += 1
            else:
                # Some but not all numbers match
                partial_overlap += 1
        
        pair_name = f"{list_names[i]}_vs_{list_names[j]}"
        results[pair_name] = {
            'full_overlap': full_overlap,
            'partial_overlap': partial_overlap,
            'no_overlap': no_overlap,
            'total': n_samples
        }
    
    return results

# Define list names in the correct order
list_names = ["llm_data", "S_data", "V_data", "P_data", "swits_data"]
llm_data = [str(x) for x in df['LLM'].tolist()]
S_data = [str(x) for x in df['Sara '].tolist()]
V_data = [str(x) for x in df['Valtteri'].tolist()]
P_data = [str(x) for x in df['Paul'].tolist()]
swits_data = [str(x) for x in df['merged'].tolist()]
lists = [llm_data, S_data, V_data, P_data, swits_data]

result = count_positional_overlaps(lists, list_names)

print("Position-wise overlapping analysis:")
print("=" * 70)
for pair, counts in result.items():
    total = counts['total']
    print(f"{pair}:")
    print(f"  Full overlap:    {counts['full_overlap']}/{total} ({counts['full_overlap']/total:.3f})")
    print(f"  Partial overlap: {counts['partial_overlap']}/{total} ({counts['partial_overlap']/total:.3f})")
    print(f"  No overlap:      {counts['no_overlap']}/{total} ({counts['no_overlap']/total:.3f})")
    print()

print("=" * 70)
print(f"Total samples analyzed: {len(llm_data)}")

Position-wise overlapping analysis:
llm_data_vs_S_data:
  Full overlap:    38/79 (0.481)
  Partial overlap: 16/79 (0.203)
  No overlap:      25/79 (0.316)

llm_data_vs_V_data:
  Full overlap:    19/79 (0.241)
  Partial overlap: 16/79 (0.203)
  No overlap:      44/79 (0.557)

llm_data_vs_P_data:
  Full overlap:    24/79 (0.304)
  Partial overlap: 32/79 (0.405)
  No overlap:      23/79 (0.291)

llm_data_vs_swits_data:
  Full overlap:    17/79 (0.215)
  Partial overlap: 29/79 (0.367)
  No overlap:      33/79 (0.418)

S_data_vs_V_data:
  Full overlap:    20/79 (0.253)
  Partial overlap: 36/79 (0.456)
  No overlap:      23/79 (0.291)

S_data_vs_P_data:
  Full overlap:    28/79 (0.354)
  Partial overlap: 41/79 (0.519)
  No overlap:      10/79 (0.127)

S_data_vs_swits_data:
  Full overlap:    16/79 (0.203)
  Partial overlap: 35/79 (0.443)
  No overlap:      28/79 (0.354)

V_data_vs_P_data:
  Full overlap:    17/79 (0.215)
  Partial overlap: 42/79 (0.532)
  No overlap:      20/79 (0.253)

V_da

In [142]:
list_names = ["llm_data", "S_data", "V_data", "P_data","swits_data"]

S_data = [str(x) for x in df['Sara '].tolist()]
V_data = [str(x) for x in df['Valtteri'].tolist()]
P_data = [str(x) for x in df['Paul'].tolist()]
swits_data = [str(x) for x in df['merged'].tolist()]
llm_data = [str(x) for x in df['LLM'].tolist()]
lists = [llm_data, S_data, V_data, P_data,swits_data]
result = count_positional_overlaps(lists, list_names)

print("Position-wise overlapping counts:")
print("=" * 50)
for pair, count in result.items():
    print(f"{pair}: {count/len(llm_data)} positions")

print("=" * 50)
print(f"Total samples analyzed: {len(llm_data)}")

Position-wise overlapping counts:
llm_data_vs_S_data: 0.6835443037974683 positions
llm_data_vs_V_data: 0.4430379746835443 positions
llm_data_vs_P_data: 0.7088607594936709 positions
llm_data_vs_swits_data: 0.5822784810126582 positions
S_data_vs_V_data: 0.7088607594936709 positions
S_data_vs_P_data: 0.8734177215189873 positions
S_data_vs_swits_data: 0.6455696202531646 positions
V_data_vs_P_data: 0.7468354430379747 positions
V_data_vs_swits_data: 0.5949367088607594 positions
P_data_vs_swits_data: 0.7848101265822784 positions
Total samples analyzed: 79


In [138]:
# Take the average over the three SWITS annotators 
print("Position-wise overlapping counts, only pay attention to SWITS numbers:")
swits_data = [str(x) for x in df['Annotator 1'].tolist()]
lists = [llm_data, S_data, V_data, P_data,swits_data]
result = count_positional_overlaps(lists, list_names)

swits_data = [str(x) for x in df['Annotator 2 '].tolist()]
lists = [llm_data, S_data, V_data, P_data,swits_data]
result2 = count_positional_overlaps(lists, list_names)

swits_data = [str(x) for x in df['Annotator 3 '].tolist()]
lists = [llm_data, S_data, V_data, P_data,swits_data]
result3 = count_positional_overlaps(lists, list_names)
print("=" * 50)
for (pair, count),(pair2, count2),(pair3, count3) in zip(result.items(), result2.items(), result3.items()):
    res1 = count/len(llm_data)
    res2 = count2/len(llm_data)
    res3 = count3/len(llm_data)
    avg = np.mean([res1, res2, res3])
    print(f"{pair}: {avg*100} positions")
    #print(f"{pair2}: {count2/len(llm_data)} positions")
    #print(f"{pair3}: {count3/len(llm_data)} positions")

print("=" * 50)
print(f"Total samples analyzed: {len(llm_data)}")

Position-wise overlapping counts, only pay attention to SWITS numbers:
llm_data_vs_S_data: 68.35443037974683 positions
llm_data_vs_V_data: 44.303797468354425 positions
llm_data_vs_P_data: 70.88607594936708 positions
llm_data_vs_swits_data: 54.852320675105496 positions
S_data_vs_V_data: 70.88607594936708 positions
S_data_vs_P_data: 87.34177215189872 positions
S_data_vs_swits_data: 62.02531645569619 positions
V_data_vs_P_data: 74.68354430379746 positions
V_data_vs_swits_data: 58.64978902953587 positions
P_data_vs_swits_data: 77.21518987341773 positions
Total samples analyzed: 79


In [146]:
def compute_agreement(df, row_n_1, row_n_2,row_n_3=None): 
    """ 
    Compute agreement (Cohen's kappa and Krippendorff's Alpha) of provided dataframe
    between row1 and row2 and possibly row3. 
    """
    annots = []
    for idx, row in df.iterrows():
        annot_id = str.zfill(str(idx), 3)
        annot_coder1 = ['Annotator 1 ', annot_id, create_annot(str(row[row_n_1]))]
        annot_coder2 = ['Annotator 2', annot_id, create_annot(str(row[row_n_2]))]
        annots.append(annot_coder1)
        annots.append(annot_coder2)
        if row_n_3 != None: 
            annot_coder3 = ['Annotator 3', annot_id, create_annot(str(row[row_n_3]))]
            annots.append(annot_coder3)

    task = agreement.AnnotationTask(distance=jaccard_distance)
    task.load_array(annots)
    #print("Cohen's Kappa: {}".format(task.kappa()))
    #print("Krippendorff's Alpha: {}".format(task.alpha()))
    return task.kappa()

print("Sara vs SWITS merged")
compute_agreement(df, 4, 8)

print("SWITS 1 vs S")
print(np.mean([compute_agreement(df, 1, 4),compute_agreement(df, 2, 4),compute_agreement(df, 3, 4)]))
print("SWITS 1 vs V")
print(np.mean([compute_agreement(df, 1, 5),compute_agreement(df, 2, 5),compute_agreement(df, 3, 5)]))
print("SWITS 1 vs P")
print(np.mean([compute_agreement(df, 1, 6),compute_agreement(df, 2, 6),compute_agreement(df, 3, 6)]))
print("SWITS 1 vs AI")
print(np.mean([compute_agreement(df, 1, 7),compute_agreement(df, 2, 7),compute_agreement(df, 3, 7)]))

Sara vs SWITS merged
SWITS 1 vs S
0.252554349731541
SWITS 1 vs V
0.21849101193726114
SWITS 1 vs P
0.2955823240038161
SWITS 1 vs AI
0.18174247002824592


In [147]:
from nltk import agreement
from nltk.metrics.distance import masi_distance
from nltk.metrics.distance import jaccard_distance

def create_annot(an):
    """
    Create frozensets from comma-separated labels or single labels.
    Handles empty values and converts all labels to strings.
    """
    if pd.isna(an) or an == '':
        return frozenset()
    an = str(an)
    if "," in an:
        # Split on commas and strip whitespace from each label
        return frozenset(label.strip() for label in an.split(","))
    else:
        # Single label
        return frozenset([an.strip()])

def compute_agreement(df, row_n_1, row_n_2,row_n_3=None): 
    """ 
    Compute agreement (Cohen's kappa and Krippendorff's Alpha) of provided dataframe
    between row1 and row2 and possibly row3. 
    """
    annots = []
    for idx, row in df.iterrows():
        annot_id = str.zfill(str(idx), 3)
        annot_coder1 = ['Annotator 1 ', annot_id, create_annot(str(row[row_n_1]))]
        annot_coder2 = ['Annotator 2', annot_id, create_annot(str(row[row_n_2]))]
        annots.append(annot_coder1)
        annots.append(annot_coder2)
        if row_n_3 != None: 
            annot_coder3 = ['Annotator 3', annot_id, create_annot(str(row[row_n_3]))]
            annots.append(annot_coder3)

    task = agreement.AnnotationTask(distance=jaccard_distance)
    task.load_array(annots)
    print("Cohen's Kappa: {}".format(task.kappa()))
    print("Krippendorff's Alpha: {}".format(task.alpha()))
    return task.kappa()


print("SWITS annotators")
compute_agreement(df, 1, 2, 3)
print("Sara vs Valtteri")
compute_agreement(df, 4, 5)
print("Sara vs Paul")
compute_agreement(df, 4, 6)
print("Valtteri vs Paul")
compute_agreement(df, 5, 6)
print("Sara vs AI")
compute_agreement(df, 4, 7)
print("Valtteri vs AI")
compute_agreement(df, 5, 7)
print("AI vs Paul")
compute_agreement(df, 6, 7)
#print("Sara vs SWITS merged")
#compute_agreement(df, 4, 8)
#print("Valtteri vs SWITS merged")
#compute_agreement(df, 5, 8)
#print("SWITS merged vs Paul")
#compute_agreement(df, 6, 8)
#print("SWITS merged vs AI")
#compute_agreement(df, 7, 8)

SWITS annotators
Cohen's Kappa: 0.20459789667874353
Krippendorff's Alpha: 0.04005954366548192
Sara vs Valtteri
Cohen's Kappa: 0.42236822338647584
Krippendorff's Alpha: 0.3429463754065474
Sara vs Paul
Cohen's Kappa: 0.5312437877063727
Krippendorff's Alpha: 0.4557763613083873
Valtteri vs Paul
Cohen's Kappa: 0.38058852483376054
Krippendorff's Alpha: 0.27775330495921824
Sara vs AI
Cohen's Kappa: 0.4308636276864603
Krippendorff's Alpha: 0.38968262149535904
Valtteri vs AI
Cohen's Kappa: 0.27134220743205767
Krippendorff's Alpha: 0.19277513924965195
AI vs Paul
Cohen's Kappa: 0.33551192702136096
Krippendorff's Alpha: 0.25565949134819643


0.33551192702136096